# Cadastro de Revendas de Combustíveis — Perfil Exploratório

Análise do cadastro de postos autorizados pela ANP.

**Trusted:** `data/trusted/cadastro-revendas-combustiveis/revendas.parquet` (46.095 postos)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path.cwd().parents[2]
TRUSTED = REPO / 'data' / 'trusted'

df = pd.read_parquet(TRUSTED / 'cadastro-revendas-combustiveis' / 'revendas.parquet')
print(f'Shape: {df.shape}')
print(f'CNPJs únicos: {df.cnpj.nunique():,}')
print(f'UFs: {df.uf.nunique()}')
df.head()

## 1. Distribuição por UF

In [ ]:
uf_counts = df['uf'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
uf_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Postos autorizados por UF')
ax.set_ylabel('Qtd postos')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(f'\nTop 5: {uf_counts.head(5).to_dict()}')

## 2. Bandeiras — concentração de mercado

In [ ]:
band = df['bandeira'].value_counts()
total = len(df)

print(f'Total bandeiras distintas: {band.shape[0]}')
print(f'Bandeira branca: {band.iloc[0]:,} ({band.iloc[0]/total*100:.1f}%)')
print(f'Top 5 marcadas: {band.iloc[1:6].sum():,} ({band.iloc[1:6].sum()/total*100:.1f}%)')
print()

fig, ax = plt.subplots(figsize=(10, 5))
band.head(10).plot(kind='barh', ax=ax, color='coral')
ax.set_title('Top 10 bandeiras')
ax.set_xlabel('Postos')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Série de autorizações — postos novos por ano

In [ ]:
df['ano_publicacao'] = pd.to_datetime(df['data_publicacao'], errors='coerce').dt.year
novos = df['ano_publicacao'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
novos.plot(kind='bar', ax=ax, color='teal')
ax.set_title('Autorizações publicadas por ano')
ax.set_ylabel('Postos')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 4. Bandeira branca por UF

In [ ]:
branca_uf = df[df['bandeira'] == 'BANDEIRA BRANCA'].groupby('uf').size()
total_uf = df.groupby('uf').size()
pct_branca = (branca_uf / total_uf * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
pct_branca.plot(kind='bar', ax=ax, color='orange')
ax.set_title('% Bandeira Branca por UF')
ax.set_ylabel('%')
ax.axhline(pct_branca.mean(), ls='--', color='red', label=f'Média: {pct_branca.mean():.1f}%')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Join com preços LPC — preço médio por bandeira

In [ ]:
lpc = pd.read_parquet(TRUSTED / 'serie-historica-precos' / 'lpc_posto.parquet')
print(f'LPC: {lpc.shape[0]:,} coletas, {lpc.cnpj.nunique():,} CNPJs')

# Join
lpc_band = lpc.merge(df[['cnpj', 'bandeira', 'uf']].rename(columns={'uf': 'uf_cad'}),
                     on='cnpj', how='inner')
print(f'Join: {lpc_band.shape[0]:,} linhas ({lpc_band.cnpj.nunique():,} CNPJs)')

# Preço médio por bandeira (gasolina)
gas = lpc_band[lpc_band['produto'].str.contains('GASOLINA', case=False, na=False)]
preco_band = gas.groupby('bandeira')['valor_venda'].agg(['mean', 'median', 'count'])
preco_band = preco_band[preco_band['count'] >= 100].sort_values('mean')
print('\nPreço médio gasolina por bandeira (>=100 coletas):')
print(preco_band.round(3).to_string())

## 6. Municípios — top 20 por densidade de postos

In [ ]:
mun = df.groupby(['municipio', 'uf']).size().reset_index(name='postos')
top_mun = mun.nlargest(20, 'postos')

fig, ax = plt.subplots(figsize=(10, 6))
labels = top_mun['municipio'].str.strip() + ' - ' + top_mun['uf']
ax.barh(labels, top_mun['postos'], color='mediumpurple')
ax.set_title('Top 20 municípios — nº de postos')
ax.set_xlabel('Postos')
ax.invert_yaxis()
plt.tight_layout()
plt.show()